In [ ]:
# Práctica 2026/27 — Clasificación de textos en Esperanto
## Tema: Religiones — Judaismo · Cristanismo · Islam
"""
**Equipo:** Marco Antonio Benali, Lucía Monteagudo y Sofía Madrazo
**Asignatura:** Minería de Textos
"""

In [ ]:
import re
import json
import time
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from dataclasses import dataclass

from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
)

RANDOM_STATE = 42
DATASET_CSV  = "dataset_esperanto_religio.csv"
DATASET_URL  = "https://raw.githubusercontent.com/TU_USUARIO/TU_REPO/main/dataset_esperanto_religio.csv"
RESULTS_JSON = "results_religio.json"
GENERAR_DATASET = True
MAX_FEATURES    = 3000
WIKI_API = "https://eo.wikipedia.org/w/api.php"
HEADERS = {"User-Agent": "CUNEF-MineriaTextos/1.0 (practica academica)"}

In [ ]:
# =============================================================================
# CONSTRUCCIÓN DEL DATASET
# Descargamos artículos de la Wikipedia en esperanto sobre cada religión,
# los dividimos en párrafos y etiquetamos cada párrafo con su clase.
# =============================================================================

# --- Limpieza de texto ------------------------------------------------------

def clean_paragraph(p: str) -> str:
    # Limpia un párrafo: quita etiquetas HTML, referencias tipo [1], URLs,
    # sustituye ";" por "," (el CSV usa ";" como separador) y normaliza espacios
    p = re.sub(r"<[^>]+>", " ", p)
    p = re.sub(r"\[[^\]]*\]", " ", p)
    p = re.sub(r"https?://\S+", " ", p)
    p = p.replace(";", ",")
    p = re.sub(r"\s+", " ", p).strip()
    return p


def paragraphs_from_html(html: str, min_len: int = 120):
    # Extrae los párrafos (<p>) de un artículo HTML y descarta los que tienen
    # menos de min_len caracteres (demasiado cortos para clasificar)
    soup = BeautifulSoup(html, "html.parser")
    # Eliminamos elementos que no aportan texto útil (notas, estilos, tablas...)
    for tag in soup.find_all(["sup", "style", "script", "figure", "table"]):
        tag.decompose()
    out = []
    for p in soup.find_all("p"):
        text = clean_paragraph(p.get_text(" ", strip=True))
        if len(text) >= min_len:
            out.append(text)
    return out


# --- Acceso a la API de Wikipedia --------------------------------------------

def wiki_search(query: str, limit: int = 10):
    # Busca en la Wikipedia en esperanto y devuelve los títulos encontrados
    params = {
        "action": "query", "list": "search",
        "srsearch": query, "srlimit": limit,
        "format": "json", "utf8": 1,
    }
    r = requests.get(WIKI_API, params=params, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return [x["title"] for x in r.json().get("query", {}).get("search", [])]


def wiki_article_html(title: str):
    # Descarga el HTML de un artículo y la lista de enlaces a otros artículos
    params = {
        "action": "parse", "page": title,
        "prop": "text|links", "format": "json", "redirects": 1,
    }
    r = requests.get(WIKI_API, params=params, headers=HEADERS, timeout=30)
    r.raise_for_status()
    data = r.json()
    if "parse" not in data:
        return "", []
    html  = data["parse"].get("text", {}).get("*", "")
    # Solo nos quedamos con enlaces a artículos (namespace 0)
    links = [l["*"] for l in data["parse"].get("links", []) if l.get("ns") == 0]
    return html, links


# --- Crawler por temática -----------------------------------------------------

def crawl_topic(seed_titles, keywords, max_articles=8):
    # Recorre la Wikipedia en anchura (BFS) partiendo de unos artículos semilla.
    # Solo sigue enlaces cuyo título contenga alguna palabra clave del tema,
    # para no alejarse de la categoría. Para al llegar a max_articles.
    visited, queue, articles = set(), list(seed_titles), {}
    while queue and len(visited) < max_articles:
        title = queue.pop(0)
        if title in visited:
            continue
        visited.add(title)
        try:
            html, links = wiki_article_html(title)
            articles[title] = html
            for link in links:
                if link in visited:
                    continue
                if any(kw.lower() in link.lower() for kw in keywords):
                    queue.append(link)
            time.sleep(0.2)  # pausa para no saturar la API
        except Exception as e:
            print(f"  Error con {title}: {e}")
    return articles


# Para cada clase: (artículos semilla, palabras clave para filtrar enlaces)
CATEGORY_SEEDS = {
    "judaismo":   (["Judaismo", "Torao", "Sinagogo", "Rabeno", "Talmudo",
                    "Hebrea lingvo", "Holokaŭsto"],
                   ["jud", "hebre", "sinagog", "tora", "raben", "talmud"]),
    "kristanismo":(["Kristanismo", "Jesuo", "Biblio", "Preĝejo", "Katolikismo",
                    "Protestantismo", "Ortodoksismo", "Papo"],
                   ["kristan", "krist", "preĝej", "bibl", "jesu", "eklez", "papo"]),
    "islamo":     (["Islamo", "Korano", "Mohamedo", "Moskeo", "Ramadano",
                    "Sunismo", "Ŝijaismo", "Hajĝo"],
                   ["islam", "koran", "mohamed", "moske", "ramadan", "sunism", "ŝija"]),
}


def build_dataset(min_len=120):
    # Genera el dataset completo: cada fila es un párrafo con su clase
    rows = []
    for label, (seeds, kws) in CATEGORY_SEEDS.items():
        print(f"[{label}] crawling {len(seeds)} semillas…")
        articles = crawl_topic(seeds, kws, max_articles=10)
        for title, html in articles.items():
            for p in paragraphs_from_html(html, min_len=min_len):
                rows.append({"text": p, "class": label})
    # Quitamos párrafos duplicados y barajamos las filas
    df = pd.DataFrame(rows).drop_duplicates(subset=["text"]).reset_index(drop=True)
    df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    return df


# --- Carga o generación del dataset -------------------------------------------
# Si GENERAR_DATASET es True se descarga de cero y se guarda en CSV.
# Si no, se intenta leer de GitHub, luego del CSV local y, si no existe,
# se genera.
if GENERAR_DATASET:
    df = build_dataset()
    df.to_csv(DATASET_CSV, sep=";", index=False, encoding="utf-8")
else:
    try:
        df = pd.read_csv(DATASET_URL, sep=";", encoding="utf-8")
    except Exception:
        try:
            df = pd.read_csv(DATASET_CSV, sep=";", encoding="utf-8")
        except FileNotFoundError:
            df = build_dataset()
            df.to_csv(DATASET_CSV, sep=";", index=False, encoding="utf-8")

# Primer vistazo: tamaño, número de párrafos por clase y algunos ejemplos
print("Shape:", df.shape)
print(df["class"].value_counts())
print(df.head(3))